# Kestrel — Pretraining Dataset Audit, EDA & BERT Topic Analysis

> **Project:** [**Kestrel**](https://github.com/MohammedKamalAlsyd/kestrel) — a modern **350M–700M parameter LLM** engineered to outperform OpenAI's **1.5B GPT-2 XL** across all core GPT-2 benchmarks (**WikiText-103**, **LAMBADA**, **HellaSwag**, and **ARC-Easy**) using **2–4× fewer parameters** and **zero benchmark contamination**.

---

### Notebook Architecture (Self-Contained Audit + Shared `kestrel.utils`)
Everything specific to this dataset audit (streaming, EDA, Gopher repetition, cross-doc boilerplate, BERT topic modeling + LLM topic labeling, JSD proximity, 13-gram contamination, and scoring) is **self-contained right here in this notebook**, organized into 7 clear sections. Only cross-notebook utilities (`kestrel.utils.notebook` and `kestrel.utils.text`) are imported from the repository.

### Table of Contents
1. **Section 1 — Setup & Shared Utilities:** Kaggle detection, automatic GitHub clone (`kestrel`), package installs, and shared `kestrel.utils` imports.
2. **Section 2 — Multi-Shard Randomized Streaming:** Unbiased random sampling across 8 remote shards per corpus + 100% of the 4 GPT-2 evaluation splits.
3. **Section 3 — Exploratory Data Analysis (EDA):** Document token-length distributions, structural quality signals (digit/symbol/uppercase/sentence length), TF-IDF signature keywords, and real document spot-checks.
4. **Section 4 — Quality & Noise Audit:** Length-normalized **Gopher/FineWeb n-gram repetition** ($n=2\dots 10$) and **cross-document 7-gram boilerplate** inspection. *(Document-level LZ4 was intentionally removed because it is confounded by document length.)*
5. **Section 5 — BERT Semantic Topic Modeling & Automated Topic Naming:** Embeds sampled docs with **`all-MiniLM-L6-v2`**, clusters into semantic topics with c-TF-IDF keywords, automatically names every topic using **`google/flan-t5-small`**, and visualizes the 2D semantic map + Dataset × Topic composition breakdown.
6. **Section 6 — Domain Proximity (JSD) vs. 13-Gram Contamination:** Base-2 **Unigram & Bigram Jensen-Shannon Divergence** against `OpenWebText` and all 4 benchmarks paired with exact **13-gram overlap** leakage checks.
7. **Section 7 — Final Ranked Scorecard & 2-Stage WSD Mixture:** Length-unbiased composite ranking (`40% Cleanliness + 45% JSD Alignment + 15% Decontamination`), contamination verification, and export to CSV/JSON.

### Plain-English Metric Glossary

| Metric / Technique | Plain-English Meaning | Why It Matters |
| :--- | :--- | :--- |
| **Multi-Shard Random Sampling** | Randomly choosing `8` remote shards across the full dataset + shuffling inside each shard (`buffer_size=2,000`). | Prevents opening-shard prefix bias (date/domain sorting) without downloading terabytes on Kaggle. |
| **Gopher Repetition ($n=2\dots 10$)** | Fraction of a document's characters covered by repeated 2–10-word phrases. | Length-normalized (fair to both short web pages and 3,000-token PDFs). |
| **Cross-Doc 7-Gram Boilerplate** | 7-word phrases repeated across `>= 0.5%` (min 10) of distinct documents. | Catches site-wide cookie banners & footers that appear once per page. |
| **BERT Topic Modeling + `flan-t5`** | Clustering `all-MiniLM-L6-v2` semantic embeddings + naming each topic via `google/flan-t5-small`. | Shows *what subjects* each corpus actually teaches (STEM vs. News vs. Code vs. Everyday life). |
| **Unigram & Bigram JSD** | Base-2 Jensen-Shannon Divergence (`0.0` = identical, `1.0` = disjoint) between word/phrase distributions. | Measures how closely a dataset's English register matches GPT-2 (`OpenWebText`) and target benchmarks. |
| **13-Gram Contamination** | Exact 13-word overlap between training documents and evaluation test splits. | Separates **healthy domain proximity** from **memorizing test questions**. |


---
## Section 1 — Environment Setup, GitHub Repo Sync & Shared Utilities

Auto-detects **Kaggle** (`/kaggle/working`) vs. local execution, installs required packages (including `sentence-transformers`, `transformers`, and `scikit-learn` for BERT topic modeling & `flan-t5-small` topic naming), clones **`https://github.com/MohammedKamalAlsyd/kestrel.git`** (default branch `master`), and imports our reusable shared utilities from **`kestrel.utils`**.

> **Kaggle Tip:** Enable **Internet ON** and (recommended for ~10× faster BERT embeddings) turn on a **GPU T4 x2 / P100** accelerator in the Kaggle sidebar. Everything also runs cleanly on CPU!


In [ ]:
import gc
import itertools
import json
import math
import os
import random
import re
import shutil
import subprocess
import sys
import time
from collections import Counter, defaultdict
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Iterator, Sequence

# ---------------------------------------------------------------------------
# 1. Detect Kaggle vs. Local Workspace
# ---------------------------------------------------------------------------
REPO_URL = "https://github.com/MohammedKamalAlsyd/kestrel.git"

IS_KAGGLE = Path("/kaggle/working").exists()
WORK_DIR = Path("/kaggle/working") if IS_KAGGLE else Path.cwd()
SAMPLES_DIR = WORK_DIR / "samples"
FIGURES_DIR = SAMPLES_DIR / "figures"
CACHE_DIR = WORK_DIR / ".hf_cache"
SAMPLES_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------------------------
# 2. Install Missing Dependencies Quietly
# ---------------------------------------------------------------------------
REQUIRED_PACKAGES = [
    ("datasets", "datasets>=3.0.0"),
    ("tiktoken", "tiktoken>=0.7.0"),
    ("scipy", "scipy>=1.12"),
    ("zstandard", "zstandard>=0.22"),
    ("seaborn", "seaborn>=0.13"),
    ("sklearn", "scikit-learn>=1.4"),
    ("sentence_transformers", "sentence-transformers>=3.0"),
    ("transformers", "transformers>=4.40"),
]
missing = []
for mod_name, pip_spec in REQUIRED_PACKAGES:
    try:
        __import__(mod_name)
    except ImportError:
        missing.append(pip_spec)

if missing:
    print("Installing missing packages:", ", ".join(missing))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

# ---------------------------------------------------------------------------
# 3. Locate or Auto-Clone Kestrel Repo (for shared kestrel.utils + docs/images)
# ---------------------------------------------------------------------------
REPO_ROOT = None
for candidate_root in [
    Path.cwd(),
    Path.cwd().parent,
    Path("/kaggle/input/kestrel"),
    WORK_DIR / "kestrel",
]:
    if (candidate_root / "src" / "kestrel" / "utils" / "notebook.py").exists():
        REPO_ROOT = candidate_root.resolve()
        break

if REPO_ROOT is None:
    clone_dir = WORK_DIR / "kestrel"
    if (clone_dir / ".git").exists():
        print(f"Updating Kestrel repo at {clone_dir} ...")
        subprocess.check_call(["git", "-C", str(clone_dir), "pull", "--ff-only"])
    else:
        if clone_dir.exists() and any(clone_dir.iterdir()):
            shutil.rmtree(clone_dir, ignore_errors=True)
        print(f"Cloning Kestrel repo from {REPO_URL} -> {clone_dir} ...")
        res = subprocess.run(
            ["git", "clone", "--depth", "1", REPO_URL, str(clone_dir)],
            capture_output=True,
            text=True,
        )
        if res.returncode != 0:
            raise RuntimeError(
                f"git clone failed (exit {res.returncode}). "
                f"Ensure Kaggle Internet is ON and repo is public/accessible.\n"
                f"stderr: {res.stderr.strip()}"
            )
    REPO_ROOT = clone_dir.resolve()

src_path = str((REPO_ROOT / "src").resolve())
if src_path not in sys.path:
    sys.path.insert(0, src_path)

# ---------------------------------------------------------------------------
# 4. Import Shared Cross-Notebook Utilities (kestrel.utils)
# ---------------------------------------------------------------------------
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from IPython.display import display
from scipy.spatial.distance import jensenshannon

from kestrel.utils.notebook import (
    configure_hf_cache,
    get_hf_token,
    show_explainer_visual,
)
from kestrel.utils.text import (
    count_gpt2_tokens,
    extract_ngrams_set,
    get_gpt2_tokenizer,
    normalize_alnum_words,
    normalize_words,
)

sns.set_theme(style="whitegrid", context="notebook")
configure_hf_cache(CACHE_DIR)
HF_TOKEN = get_hf_token("HF_TOKEN")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ---------------------------------------------------------------------------
# 5. Experiment Configuration
# ---------------------------------------------------------------------------
TARGET_TOKENS = 3_500_000     # ~3.5M GPT-2 BPE tokens per candidate dataset
MAX_DOCS = 5_000              # Cap at 5,000 docs per candidate dataset
RANDOM_SAMPLE = True          # Multi-shard random sampling (avoids opening-shard prefix bias)
NUM_SHARDS_TO_SAMPLE = 8      # Sample evenly across 8 randomly chosen remote shards
SHUFFLE_BUFFER_SIZE = 2_000   # Per-shard shuffle buffer size
SEED = 42
FORCE_REFRESH = False         # Set True to re-stream & overwrite cached JSONL files

# BERT Topic Modeling Config
BERT_EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
TOPIC_NAMER_MODEL = "google/flan-t5-small"
N_TOPICS = 12
TOPIC_DOCS_PER_DATASET = 500  # Up to 500 docs/dataset (~2,500-3,000 total; ~10s on T4 GPU)

print(f"[OK] Shared kestrel.utils loaded from: {REPO_ROOT}")
print(f"Environment : {'Kaggle' if IS_KAGGLE else 'Local'} | Device: {DEVICE.upper()} | Work Dir: {WORK_DIR}")
print(f"HF_TOKEN    : {'Detected (gated Nemotron-CC enabled)' if HF_TOKEN else 'Not set (ungated datasets only)'}")
print(f"Sampling    : {NUM_SHARDS_TO_SAMPLE} random shards (buf={SHUFFLE_BUFFER_SIZE}) | <= {MAX_DOCS:,} docs or {TARGET_TOKENS:,} tokens")


In [ ]:
# Uses shared kestrel.utils.notebook helper (local docs/images/ first -> GitHub raw fallback)
show_explainer_visual("01_kestrel_data_pipeline.png", "Figure 1: End-to-End Kestrel Data Selection Funnel — Multi-Shard Sampling, EDA, BERT Topic Modeling, Quality/JSD/13-Gram Audits & 2-Stage Mix.")


[Visual: Figure 1: End-to-End Kestrel Data Selection Funnel — Multi-Shard Sampling, EDA, BERT Topic Modeling, Quality/JSD/13-Gram Audits & 2-Stage Mix. — https://raw.githubusercontent.com/MohammedKamalAlsyd/kestrel/master/docs/images/01_kestrel_data_pipeline.png]

---
## Section 2 — Candidate Registry & Multi-Shard Randomized Streaming

### Why Multi-Shard Random Sampling Instead of Taking the Opening Prefix?
Reading only the first 5,000 rows of shard `00000` introduces **prefix bias** (e.g., if shards are grouped by crawl date or URL domain). Our self-contained `stream_matched_slice()` below selects **`8` distinct remote shards uniformly at random** across each dataset's full shard list, shuffles inside each shard with a `2,000`-row buffer, and draws an equal token quota (`~3.5M / 8`) from each shard — caching lightweight ~20 MB JSONL files in `samples/`.

### The 6 Candidate Datasets + 4 GPT-2 Benchmarks
1. **`FineWeb-Edu (10BT)`** — Educational classifier-filtered Common Crawl *(ODC-By v1.0)*.
2. **`DCLM-Baseline 1.0`** — FastText-filtered broad web corpus *(CC-BY-4.0)*.
3. **`Nemotron-CC v2.1 (HQ)`** — NVIDIA HQ + synthetic web corpus *(Gated; requires `HF_TOKEN`)*.
4. **`Nemotron-ClimbMix (400B)`** — 2025 NeurIPS CLIMB cluster-bootstrapped sub-1B mix *(Upstream CC-BY-NC-4.0)*.
5. **`FinePDFs-Edu (eng_Latn)`** — 2025–2026 long-form educational PDF extraction *(ODC-By v1.0)*.
6. **`OpenWebText` (Control)** — GPT-2 original distribution anchor (**control only**).

**Evaluation Benchmarks (100% loaded):** `HellaSwag (val)`, `ARC-Easy (test)`, `LAMBADA (test)`, `WikiText-103 (test)`.


In [ ]:
# Uses shared kestrel.utils.notebook helper (local docs/images/ first -> GitHub raw fallback)
show_explainer_visual("anim_01_streaming_and_13gram_scan.gif", "Animation 1: Low-RAM Equal-Token Streaming (Top) & 13-Gram Sliding-Window Contamination Scanner (Bottom).")


[Visual: Animation 1: Low-RAM Equal-Token Streaming (Top) & 13-Gram Sliding-Window Contamination Scanner (Bottom). — https://raw.githubusercontent.com/MohammedKamalAlsyd/kestrel/master/docs/images/anim_01_streaming_and_13gram_scan.gif]

In [ ]:
from datasets import load_dataset


@dataclass(frozen=True)
class DatasetSpec:
    key: str
    display_name: str
    repo_id: str
    config: str | None = None
    split: str = "train"
    text_field: str = "text"
    license: str = "Unknown"
    gated: bool = False
    is_control: bool = False
    load_kwargs: dict[str, Any] = field(default_factory=dict)
    fallbacks: tuple[dict[str, Any], ...] = ()


@dataclass(frozen=True)
class EvalSpec:
    key: str
    display_name: str
    repo_id: str
    config: str | None = None
    split: str = "test"
    kind: str = "text"


CANDIDATE_DATASETS: dict[str, DatasetSpec] = {
    "fineweb_edu": DatasetSpec(
        key="fineweb_edu",
        display_name="FineWeb-Edu (10BT)",
        repo_id="HuggingFaceFW/fineweb-edu",
        config="sample-10BT",
        license="ODC-By v1.0",
    ),
    "dclm_baseline": DatasetSpec(
        key="dclm_baseline",
        display_name="DCLM-Baseline 1.0",
        repo_id="mlfoundations/dclm-baseline-1.0-parquet",
        license="CC-BY-4.0",
        fallbacks=(
            {
                "repo_id": "json",
                "config": None,
                "split": "train",
                "load_kwargs": {
                    "data_files": "hf://datasets/mlfoundations/dclm-baseline-1.0/**/*_processed.jsonl.zst"
                },
            },
        ),
    ),
    "nemotron_cc": DatasetSpec(
        key="nemotron_cc",
        display_name="Nemotron-CC v2.1 (HQ)",
        repo_id="nvidia/Nemotron-CC-v2.1",
        license="NVIDIA Data Agreement for Model Training",
        gated=True,
        load_kwargs={"data_dir": "High-Quality"},
        fallbacks=(
            {"repo_id": "nvidia/Nemotron-CC-v2", "config": None, "split": "train", "load_kwargs": {"data_dir": "High-Quality"}},
            {"repo_id": "nvidia/Nemotron-Pretraining-Dataset-sample", "config": "Nemotron-CC-v2-High-Quality", "split": "train", "load_kwargs": {}},
        ),
    ),
    "climbmix": DatasetSpec(
        key="climbmix",
        display_name="Nemotron-ClimbMix (400B)",
        repo_id="karpathy/climbmix-400b-shuffle",
        license="CC-BY-NC-4.0 (upstream NVIDIA; MIT mirror)",
        fallbacks=({"repo_id": "OptimalScale/ClimbMix", "config": None, "split": "train", "load_kwargs": {}},),
    ),
    "finepdfs_edu": DatasetSpec(
        key="finepdfs_edu",
        display_name="FinePDFs-Edu (eng_Latn)",
        repo_id="HuggingFaceFW/finepdfs-edu",
        config="eng_Latn",
        license="ODC-By v1.0",
        fallbacks=({"repo_id": "HuggingFaceFW/finepdfs", "config": "eng_Latn", "split": "train", "load_kwargs": {}},),
    ),
    "openwebtext": DatasetSpec(
        key="openwebtext",
        display_name="OpenWebText (GPT-2 Control)",
        repo_id="Skylion007/openwebtext",
        license="Public Domain / CC0",
        is_control=True,
        fallbacks=(
            {
                "repo_id": "parquet",
                "config": None,
                "split": "train",
                "load_kwargs": {
                    "data_files": "hf://datasets/Skylion007/openwebtext@refs/convert/parquet/plain_text/train/*.parquet"
                },
            },
            {"repo_id": "stas/openwebtext-10k", "config": None, "split": "train", "load_kwargs": {}},
        ),
    ),
}

EVAL_BENCHMARKS: dict[str, EvalSpec] = {
    "hellaswag": EvalSpec("hellaswag", "HellaSwag (val)", "Rowan/hellaswag", None, "validation", "hellaswag"),
    "arc_easy": EvalSpec("arc_easy", "ARC-Easy (test)", "allenai/ai2_arc", "ARC-Easy", "test", "arc"),
    "lambada": EvalSpec("lambada", "LAMBADA (test)", "EleutherAI/lambada_openai", "en", "test", "lambada"),
    "wikitext103": EvalSpec("wikitext103", "WikiText-103 (test)", "Salesforce/wikitext", "wikitext-103-raw-v1", "test", "wikitext"),
}


def load_jsonl_slice(path: Path) -> list[dict[str, Any]]:
    with path.open("r", encoding="utf-8") as fh:
        return [json.loads(line) for line in fh if line.strip()]


def _open_hf_iterable(spec: DatasetSpec, hf_token: str | None = None) -> Any:
    attempts = [
        {"repo_id": spec.repo_id, "config": spec.config, "split": spec.split, "load_kwargs": dict(spec.load_kwargs)},
        *[dict(fb) for fb in spec.fallbacks],
    ]
    last_exc = None
    for attempt in attempts:
        kwargs = {"split": attempt.get("split", spec.split), "streaming": True, **attempt.get("load_kwargs", {})}
        if hf_token:
            kwargs["token"] = hf_token
        try:
            cfg = attempt.get("config")
            ds = load_dataset(attempt["repo_id"], cfg, **kwargs) if cfg is not None else load_dataset(attempt["repo_id"], **kwargs)
            _ = next(iter(ds))
            return ds
        except Exception as exc:
            last_exc = exc
    raise RuntimeError(f"Failed to stream '{spec.key}': {last_exc}") from last_exc


def stream_matched_slice(
    spec: DatasetSpec,
    cache_path: Path,
    target_tokens: int = TARGET_TOKENS,
    max_docs: int = MAX_DOCS,
    min_chars: int = 64,
    hf_token: str | None = HF_TOKEN,
    force_refresh: bool = FORCE_REFRESH,
    random_sample: bool = RANDOM_SAMPLE,
    num_shards_to_sample: int = NUM_SHARDS_TO_SAMPLE,
    shuffle_buffer_size: int = SHUFFLE_BUFFER_SIZE,
    seed: int = SEED,
) -> list[dict[str, Any]]:
    if cache_path.exists() and not force_refresh:
        return load_jsonl_slice(cache_path)

    enc = get_gpt2_tokenizer()
    ds = _open_hf_iterable(spec, hf_token=hf_token)

    # Multi-shard random sampling across the entire corpus to eliminate prefix bias
    n_shards = int(getattr(ds, "n_shards", 1) or 1)
    if random_sample and n_shards > 1 and hasattr(ds, "shard"):
        k = min(max(1, num_shards_to_sample), n_shards)
        chosen = random.Random(seed).sample(range(n_shards), k)
        shard_iters = []
        for shard_idx in chosen:
            sds = ds.shard(num_shards=n_shards, index=shard_idx)
            if shuffle_buffer_size > 1 and hasattr(sds, "shuffle"):
                sds = sds.shuffle(seed=seed + shard_idx, buffer_size=shuffle_buffer_size)
            shard_iters.append(iter(sds))
    elif random_sample and shuffle_buffer_size > 1 and hasattr(ds, "shuffle"):
        shard_iters = [iter(ds.shuffle(seed=seed, buffer_size=shuffle_buffer_size))]
    else:
        shard_iters = [iter(ds)]

    records: list[dict[str, Any]] = []
    total_tokens = 0
    cache_path.parent.mkdir(parents=True, exist_ok=True)
    tmp_file = cache_path.with_suffix(".jsonl.tmp")

    with tmp_file.open("w", encoding="utf-8") as fh:
        for pos, row_iter in enumerate(shard_iters):
            rem_shards = len(shard_iters) - pos
            rem_tokens = max(0, target_tokens - total_tokens)
            rem_docs = max(0, max_docs - len(records))
            if rem_tokens == 0 or rem_docs == 0:
                break
            tok_quota = math.ceil(rem_tokens / rem_shards)
            doc_quota = math.ceil(rem_docs / rem_shards)
            s_toks = 0
            s_docs = 0

            for row in row_iter:
                raw = row.get(spec.text_field)
                if not isinstance(raw, str):
                    continue
                text = raw.strip()
                if len(text) < min_chars:
                    continue
                n_tok = count_gpt2_tokens(text, enc)
                if n_tok == 0:
                    continue
                rec = {"text": text, "n_tokens": n_tok}
                records.append(rec)
                fh.write(json.dumps(rec, ensure_ascii=False) + "\n")
                total_tokens += n_tok
                s_toks += n_tok
                s_docs += 1
                if len(records) >= max_docs or total_tokens >= target_tokens or s_docs >= doc_quota or s_toks >= tok_quota:
                    break

    tmp_file.replace(cache_path)
    return records


def load_eval_benchmark_texts(spec: EvalSpec, cache_path: Path, force_refresh: bool = FORCE_REFRESH) -> list[str]:
    if cache_path.exists() and not force_refresh:
        return [r["text"] for r in load_jsonl_slice(cache_path)]

    ds = load_dataset(spec.repo_id, spec.config, split=spec.split) if spec.config else load_dataset(spec.repo_id, split=spec.split)
    texts: list[str] = []
    if spec.kind == "hellaswag":
        for row in ds:
            ctx = str(row.get("ctx", "")).strip()
            endings = row.get("endings", []) or []
            lbl = row.get("label", "")
            idx = lbl if isinstance(lbl, int) else (int(lbl) if isinstance(lbl, str) and lbl.isdigit() else -1)
            gold = endings[idx] if 0 <= idx < len(endings) else " ".join(endings)
            if f"{ctx} {gold}".strip():
                texts.append(f"{ctx} {gold}".strip())
    elif spec.kind == "arc":
        for row in ds:
            q = str(row.get("question", "")).strip()
            choices = (row.get("choices") or {}).get("text", [])
            if f"{q} {' '.join(str(c) for c in choices)}".strip():
                texts.append(f"{q} {' '.join(str(c) for c in choices)}".strip())
    elif spec.kind == "lambada":
        texts = [str(r.get("text", "")).strip() for r in ds if str(r.get("text", "")).strip()]
    elif spec.kind == "wikitext":
        cur: list[str] = []
        for row in ds:
            line = str(row.get("text", "")).strip()
            if not line:
                continue
            if line.startswith("= ") and line.endswith(" =") and not line.startswith("= =") and cur:
                texts.append("\n".join(cur))
                cur = [line]
            else:
                cur.append(line)
        if cur:
            texts.append("\n".join(cur))

    cache_path.parent.mkdir(parents=True, exist_ok=True)
    with cache_path.open("w", encoding="utf-8") as fh:
        for t in texts:
            fh.write(json.dumps({"text": t}, ensure_ascii=False) + "\n")
    return texts


# --- Execute Streaming & Benchmark Loading ---
candidate_slices: dict[str, list[dict[str, Any]]] = {}
stream_summary_rows = []

print("=== 1/2 Multi-Shard Randomized Streaming of Candidate Datasets ===")
for key, spec in CANDIDATE_DATASETS.items():
    cache_file = SAMPLES_DIR / f"{key}.jsonl"
    t0 = time.time()
    try:
        recs = stream_matched_slice(spec, cache_file)
        candidate_slices[key] = recs
        n_tok = sum(r["n_tokens"] for r in recs)
        mb = cache_file.stat().st_size / (1024 * 1024)
        print(f"[OK]   {spec.display_name:<28s} | {len(recs):>5,d} docs | {n_tok:>9,d} tok | {mb:>5.1f} MB | {time.time()-t0:>5.1f}s")
        stream_summary_rows.append({
            "Key": key, "Dataset": spec.display_name, "License": spec.license,
            "Docs": len(recs), "GPT-2 Tokens": n_tok,
            "Mean Tok/Doc": round(n_tok / max(len(recs), 1), 1), "JSONL (MB)": round(mb, 2),
        })
    except Exception as exc:
        print(f"[SKIP] {spec.display_name}: {exc}")

print("\n=== 2/2 Loading 100% of GPT-2 Target Evaluation Benchmarks ===")
eval_texts_by_benchmark: dict[str, list[str]] = {}
for bkey, bspec in EVAL_BENCHMARKS.items():
    t0 = time.time()
    eval_texts_by_benchmark[bkey] = load_eval_benchmark_texts(bspec, SAMPLES_DIR / f"eval_{bkey}.jsonl")
    print(f"[OK]   {bspec.display_name:<22s} | {len(eval_texts_by_benchmark[bkey]):>6,d} examples | {time.time()-t0:>4.1f}s")

ds_keys = list(candidate_slices.keys())
ds_labels = {k: CANDIDATE_DATASETS[k].display_name for k in ds_keys}
palette = dict(zip(ds_keys, sns.color_palette("tab10", n_colors=max(len(ds_keys), 1))))

display(pd.DataFrame(stream_summary_rows))


---
## Section 3 — Exploratory Data Analysis (EDA): Lengths, Structure, TF-IDF Terms & Spot-Checks

Before running automated scores, **Senior AI Engineers always inspect the raw data directly**. In this section we examine:
1. **Document Token-Length Distributions (log-scale):** Reveals which datasets teach short web snippets (~600 tokens) vs. long-range multi-page reasoning (`FinePDFs-Edu` ~3,200+ tokens).
2. **Structural Quality Profile:** Average word length, average sentence length, digit %, uppercase %, symbol/punctuation %, and newline density — catching code/table/math heavy corpora vs. clean narrative prose without length bias.
3. **Signature TF-IDF Terms per Dataset:** Highlights the vocabulary each corpus uniquely over-indexes on compared to the others.
4. **Random Document Spot-Check:** Prints real excerpts from every dataset so you can judge writing quality with your own eyes.


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

SENT_SPLIT_RE = re.compile(r"[.!?]+")

eda_rows = []
length_plot_rows = []

for dkey in ds_keys:
    recs = candidate_slices[dkey]
    tok_lens = np.array([r["n_tokens"] for r in recs], dtype=np.float64)
    texts = [r["text"] for r in recs]

    word_lens, sent_lens, digit_pcts, upper_pcts, sym_pcts, nl_per_1k = [], [], [], [], [], []
    for txt in texts:
        n_chars = max(len(txt), 1)
        digits = sum(c.isdigit() for c in txt)
        uppers = sum(c.isupper() for c in txt)
        syms = sum((not c.isalnum()) and (not c.isspace()) for c in txt)
        nls = txt.count("\n")
        words = normalize_words(txt)
        sents = [s for s in SENT_SPLIT_RE.split(txt) if s.strip()]

        digit_pcts.append(100.0 * digits / n_chars)
        upper_pcts.append(100.0 * uppers / n_chars)
        sym_pcts.append(100.0 * syms / n_chars)
        nl_per_1k.append(1000.0 * nls / n_chars)
        if words:
            word_lens.append(float(np.mean([len(w) for w in words])))
            sent_lens.append(len(words) / max(len(sents), 1))

    eda_rows.append({
        "Dataset": ds_labels[dkey],
        "Docs": len(recs),
        "Median Tok/Doc": round(float(np.median(tok_lens)), 1),
        "Mean Tok/Doc": round(float(np.mean(tok_lens)), 1),
        "P90 Tok/Doc": round(float(np.percentile(tok_lens, 90)), 1),
        "P99 Tok/Doc": round(float(np.percentile(tok_lens, 99)), 1),
        "Mean Word Len": round(float(np.mean(word_lens)), 2),
        "Words / Sent": round(float(np.mean(sent_lens)), 1),
        "Digits (%)": round(float(np.mean(digit_pcts)), 2),
        "Uppercase (%)": round(float(np.mean(upper_pcts)), 2),
        "Symbols (%)": round(float(np.mean(sym_pcts)), 2),
        "Newlines / 1k Chars": round(float(np.mean(nl_per_1k)), 2),
    })
    for t_len in tok_lens:
        length_plot_rows.append({"Dataset": ds_labels[dkey], "Tokens": t_len})

eda_df = pd.DataFrame(eda_rows)
print("=== Structural & Length Profile Across Candidate Corpora ===")
display(eda_df)

# Plot 1: Log-scale Document Length Distribution + Structural Comparison
fig, (ax_len, ax_struct) = plt.subplots(1, 2, figsize=(16.5, 5.2))

len_df = pd.DataFrame(length_plot_rows)
len_df["Log10 Tokens"] = np.log10(np.clip(len_df["Tokens"], 1, None))
sns.boxplot(
    data=len_df,
    y="Dataset",
    x="Log10 Tokens",
    hue="Dataset",
    legend=False,
    palette=[palette[k] for k in ds_keys],
    ax=ax_len,
    showfliers=False,
)
tick_vals = [1.5, 2.0, 2.5, 3.0, 3.5, 4.0]
ax_len.set_xticks(tick_vals)
ax_len.set_xticklabels([f"{int(10**v):,}" for v in tick_vals])
ax_len.set_xlabel("GPT-2 Tokens per Document (Log Scale)")
ax_len.set_ylabel("")
ax_len.set_title("Document Length Distribution (Long-Context Capacity)", fontsize=12, fontweight="bold")

struct_melt = eda_df.melt(
    id_vars=["Dataset"],
    value_vars=["Digits (%)", "Uppercase (%)", "Symbols (%)"],
    var_name="Signal",
    value_name="Percent",
)
sns.barplot(data=struct_melt, y="Dataset", x="Percent", hue="Signal", ax=ax_struct, palette="Set2")
ax_struct.set_ylabel("")
ax_struct.set_yticklabels([])
ax_struct.set_xlabel("Mean Character Percentage (%)")
ax_struct.set_title("Character-Level Structural Signals (Math/Tables/Formatting)", fontsize=12, fontweight="bold")
ax_struct.legend(loc="lower right")

plt.tight_layout()
plt.savefig(FIGURES_DIR / "01_eda_length_and_structure.png", dpi=150, bbox_inches="tight")
plt.show()

# Characteristic TF-IDF Terms per Dataset (comparing corpora against each other)
corpus_docs = [" ".join(r["text"][:1500] for r in candidate_slices[k][:600]) for k in ds_keys]
tfidf = TfidfVectorizer(stop_words="english", max_features=15_000, min_df=1, max_df=0.85, token_pattern=r"(?u)\b[a-zA-Z]{3,}\b")
X_tfidf = tfidf.fit_transform(corpus_docs)
vocab_terms = np.array(tfidf.get_feature_names_out())

print("\n=== Signature TF-IDF Terms Uniquely Enriched in Each Dataset ===")
for idx, dkey in enumerate(ds_keys):
    row_vec = X_tfidf[idx].toarray().ravel()
    top_terms = vocab_terms[np.argsort(row_vec)[::-1][:10]]
    print(f"  • {ds_labels[dkey]:<28s}: {', '.join(top_terms)}")

# Random Document Spot-Checks (Read Your Data!)
print("\n=== Random Document Spot-Check (1 Sample Excerpt per Dataset) ===")
rng_spot = random.Random(SEED)
for dkey in ds_keys:
    sample_rec = rng_spot.choice(candidate_slices[dkey])
    snippet = re.sub(r"\s+", " ", sample_rec["text"])[:280]
    print(f"\n[{ds_labels[dkey]} | {sample_rec['n_tokens']:,} tokens]\n  \"{snippet}...\"")


---
## Section 4 — Quality & Noise Audit: Gopher Repetition & Cross-Doc 7-Gram Boilerplate

### Why We Dropped LZ4 Compression Ratio
In earlier tests, document-level LZ4 compression penalized long coherent documents (`FinePDFs-Edu` ~3,240 tok/doc and `DCLM-Baseline` ~1,280 tok/doc) simply because LZ4's 64 KB dictionary compresses longer prose more efficiently — even when repetition is near zero (`1.21%` on `DCLM-Baseline`)!

Instead, we rely on **two clean, length-unbiased noise audits**:
1. **Gopher / FineWeb Within-Document N-Gram Repetition ($n = 2\dots 10$):** Measures character fraction covered by repeated phrases (`top_2gram > 20%`, `top_3gram > 18%`, `top_4gram > 16%`, `dup_5gram > 15%` down to `dup_10gram > 10%`).
2. **Cross-Document 7-Gram Boilerplate:** Catches site-wide cookie banners, navigation menus, and copyright footers that appear **once** per page across `>= 0.5%` (min 10) of documents — and prints the exact phrases so you know what to strip before tokenizing.


In [ ]:
GOPHER_TOP_THRESHOLDS = {2: 0.20, 3: 0.18, 4: 0.16}
GOPHER_DUP_THRESHOLDS = {5: 0.15, 6: 0.14, 7: 0.13, 8: 0.12, 9: 0.11, 10: 0.10}


def _merged_span_chars(spans: list[tuple[int, int]], word_lens: list[int]) -> int:
    if not spans:
        return 0
    spans.sort()
    ms, me = spans[0]
    total = 0
    for s, e in spans[1:]:
        if s <= me:
            me = max(me, e)
        else:
            total += sum(word_lens[ms:me]) + max(0, me - ms - 1)
            ms, me = s, e
    total += sum(word_lens[ms:me]) + max(0, me - ms - 1)
    return total


def compute_gopher_repetition_rates(text: str) -> dict[str, Any]:
    words = normalize_alnum_words(text)
    n_words = len(words)
    if n_words < 2:
        out: dict[str, Any] = {f"top_{n}gram": 0.0 for n in (2, 3, 4)}
        out.update({f"dup_{n}gram": 0.0 for n in range(5, 11)})
        out["gopher_flagged"] = False
        return out

    word_lens = [len(w) for w in words]
    total_chars = max(sum(word_lens) + max(0, n_words - 1), 1)
    metrics: dict[str, Any] = {}
    flagged = False

    for n in (2, 3, 4):
        if n_words < n:
            val = 0.0
        else:
            pos_map: dict[tuple[str, ...], list[int]] = defaultdict(list)
            for i in range(n_words - n + 1):
                pos_map[tuple(words[i : i + n])].append(i)
            rep = {ng: ps for ng, ps in pos_map.items() if len(ps) >= 2}
            if not rep:
                val = 0.0
            else:
                best = max(rep.values(), key=lambda ps: sum(sum(word_lens[i : i + n]) + (n - 1) for i in ps))
                val = min(1.0, _merged_span_chars([(i, i + n) for i in best], word_lens) / total_chars)
        metrics[f"top_{n}gram"] = float(val)
        if val > GOPHER_TOP_THRESHOLDS[n]:
            flagged = True

    for n in range(5, 11):
        if n_words < n * 2:
            val = 0.0
        else:
            pos_map = defaultdict(list)
            for i in range(n_words - n + 1):
                pos_map[tuple(words[i : i + n])].append(i)
            spans = [(i, i + n) for ps in pos_map.values() if len(ps) >= 2 for i in ps]
            val = min(1.0, _merged_span_chars(spans, word_lens) / total_chars) if spans else 0.0
        metrics[f"dup_{n}gram"] = float(val)
        if val > GOPHER_DUP_THRESHOLDS[n]:
            flagged = True

    metrics["gopher_flagged"] = flagged
    return metrics


def audit_cross_doc_boilerplate(texts: Sequence[str], min_docs: int = 10, min_frac: float = 0.005) -> dict[str, Any]:
    n_docs = len(texts)
    if n_docs == 0:
        return {"cross_doc_boilerplate_doc_rate": 0.0, "top_cross_doc_ngrams": []}
    df_7: Counter[str] = Counter()
    per_doc_7: list[set[str]] = []
    for txt in texts:
        words = normalize_words(txt)
        u7 = {" ".join(words[i : i + 7]) for i in range(len(words) - 6)} if len(words) >= 7 else set()
        per_doc_7.append(u7)
        for ng in u7:
            df_7[ng] += 1
    thr = max(min_docs, int(min_frac * n_docs))
    flagged_7 = {ng for ng, df in df_7.items() if df >= thr}
    hits = sum(1 for s in per_doc_7 if s & flagged_7)
    return {
        "cross_doc_boilerplate_doc_rate": float(hits / n_docs),
        "top_cross_doc_ngrams": [(ng, int(df)) for ng, df in df_7.most_common(10) if df >= thr],
    }


# Run Repetition & Boilerplate Audit across all datasets
quality_audits: dict[str, dict[str, Any]] = {}
ngram_keys = [f"top_{n}gram" for n in (2, 3, 4)] + [f"dup_{n}gram" for n in range(5, 11)]

t0 = time.time()
for dkey in ds_keys:
    texts = [r["text"] for r in candidate_slices[dkey]]
    rep_rows = [compute_gopher_repetition_rates(t) for t in texts]
    cross_doc = audit_cross_doc_boilerplate(texts)
    quality_audits[dkey] = {
        "gopher_mean_rates": {k: float(np.mean([r[k] for r in rep_rows])) for k in ngram_keys},
        "gopher_flag_rate": float(np.mean([1.0 if r["gopher_flagged"] else 0.0 for r in rep_rows])),
        **cross_doc,
    }
print(f"[DONE] Gopher & Cross-Doc Boilerplate audit finished in {time.time() - t0:.1f}s")

# Plot: Gopher N-Gram Repetition Curves + Summary Flag Comparison
fig, (ax_rep, ax_bar) = plt.subplots(1, 2, figsize=(16, 5.2))
x_labels = ["top-2", "top-3", "top-4", "dup-5", "dup-6", "dup-7", "dup-8", "dup-9", "dup-10"]
for dkey in ds_keys:
    qa = quality_audits[dkey]
    vals = [100.0 * qa["gopher_mean_rates"][k] for k in ngram_keys]
    ax_rep.plot(x_labels, vals, marker="o", lw=2.2, color=palette[dkey], label=f"{ds_labels[dkey]} (flag={100*qa['gopher_flag_rate']:.1f}%)")

ax_rep.set_title("Gopher / FineWeb Within-Document Repetition (%)", fontsize=12, fontweight="bold")
ax_rep.set_xlabel("Gopher N-Gram Metric (n = 2..10)")
ax_rep.set_ylabel("Mean Document Character Fraction (%)")
ax_rep.legend(fontsize=8.5, loc="upper right")

y_pos = np.arange(len(ds_keys))
h = 0.36
gopher_pcts = [100.0 * quality_audits[k]["gopher_flag_rate"] for k in ds_keys]
cross_pcts = [100.0 * quality_audits[k]["cross_doc_boilerplate_doc_rate"] for k in ds_keys]
ax_bar.barh(y_pos - h / 2, gopher_pcts, height=h, color="#e11d48", label="Gopher Repetition Flag (%)")
ax_bar.barh(y_pos + h / 2, cross_pcts, height=h, color="#d97706", label="Cross-Doc 7-Gram Boilerplate (%)")
ax_bar.set_yticks(y_pos)
ax_bar.set_yticklabels([ds_labels[k] for k in ds_keys], fontweight="bold")
ax_bar.invert_yaxis()
ax_bar.set_xlabel("Flagged Documents (%) — Lower is Cleaner")
ax_bar.set_title("Length-Unbiased Noise Profile (Gopher + Cross-Doc 7g)", fontsize=12, fontweight="bold")
ax_bar.legend(loc="lower right")

plt.tight_layout()
plt.savefig(FIGURES_DIR / "02_gopher_and_boilerplate.png", dpi=150, bbox_inches="tight")
plt.show()

print("\n=== Top Cross-Document 7-Gram Boilerplate Phrases (Actionable Strip List) ===")
for dkey in ds_keys:
    qa = quality_audits[dkey]
    print(f"\n{ds_labels[dkey]} ({100 * qa['cross_doc_boilerplate_doc_rate']:.2f}% docs hit):")
    if qa["top_cross_doc_ngrams"]:
        for ng, df in qa["top_cross_doc_ngrams"][:3]:
            print(f'   ({df:>4d} docs) "{ng}"')
    else:
        print("   [CLEAN] No recurring 7-gram crossed the threshold.")


---
## Section 5 — BERT Semantic Topic Modeling & Automated Topic Naming (`MiniLM` + `flan-t5-small`)

### Why Add Topic Modeling to a Pretraining Data Audit?
Static heuristics like Gopher repetition and JSD tell you **cleanliness** and **surface vocabulary closeness**, but they can't tell you **what subjects a corpus actually teaches**!
By running **BERT semantic clustering** across all candidate datasets in a **shared embedding space**:
1. **BERT Embeddings (`sentence-transformers/all-MiniLM-L6-v2`):** Encodes up to `500` random documents per dataset into 384-dimensional unit-normalized semantic vectors.
2. **Semantic Clustering + c-TF-IDF (`KMeans`, $K = 12$):** Groups documents into 12 semantic topics and computes **class-based TF-IDF (c-TF-IDF)** to extract each topic's sharpest defining keywords and centroid-closest representative snippet.
3. **Automated Human-Readable Topic Naming (`google/flan-t5-small`):** Feeds each cluster's top c-TF-IDF keywords + centroid excerpt into a lightweight instruction-tuned generation model (~80M params) to generate crisp 2–4 word topic titles (e.g., *Physics & Energy Systems*, *Web Software Engineering*, *Healthcare & Clinical Studies*).
4. **Dataset × Topic Breakdown:** Reveals exactly *why* combining `DCLM-Baseline` + `FineWeb-Edu` + `FinePDFs-Edu` outperforms any single corpus!


In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.feature_extraction.text import CountVectorizer
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

# 1. Sample Balanced Pool of Documents Across All Audited Datasets
rng_topic = random.Random(SEED)
topic_texts: list[str] = []
topic_ds_keys: list[str] = []

for dkey in ds_keys:
    recs = candidate_slices[dkey]
    n_take = min(TOPIC_DOCS_PER_DATASET, len(recs))
    chosen = rng_topic.sample(recs, n_take)
    for rec in chosen:
        # First ~1,200 chars (~250 tokens) covers MiniLM's max sequence length cleanly & fast
        topic_texts.append(re.sub(r"\s+", " ", rec["text"][:1200]).strip())
        topic_ds_keys.append(dkey)

print(f"Encoding {len(topic_texts):,} documents with BERT ({BERT_EMBED_MODEL}) on {DEVICE.upper()} ...")
t0 = time.time()
embedder = SentenceTransformer(BERT_EMBED_MODEL, device=DEVICE)
embeddings = embedder.encode(
    topic_texts,
    batch_size=64 if DEVICE == "cuda" else 32,
    show_progress_bar=False,
    normalize_embeddings=True,
)
embeddings = np.asarray(embeddings, dtype=np.float32)
print(f"[OK] Encoded shape {embeddings.shape} in {time.time() - t0:.1f}s")

# Free embedder GPU/CPU RAM before loading the generation model
del embedder
if DEVICE == "cuda":
    torch.cuda.empty_cache()

# 2. Cluster into K Semantic Topics + Compute c-TF-IDF Keywords (BERTopic formula)
kmeans = KMeans(n_clusters=N_TOPICS, random_state=SEED, n_init="auto")
cluster_ids = kmeans.fit_predict(embeddings)
centroids = kmeans.cluster_centers_
centroids /= np.maximum(np.linalg.norm(centroids, axis=1, keepdims=True), 1e-9)

# Build one concatenated pseudo-document per cluster for c-TF-IDF
cluster_concat_docs = [
    " ".join(topic_texts[i] for i in range(len(topic_texts)) if cluster_ids[i] == k)
    for k in range(N_TOPICS)
]
cv = CountVectorizer(stop_words="english", max_features=20_000, min_df=2, token_pattern=r"(?u)\b[a-zA-Z]{3,}\b")
tf_mat = cv.fit_transform(cluster_concat_docs).toarray().astype(np.float64)
terms = np.array(cv.get_feature_names_out())

# BERTopic c-TF-IDF: tf(t, c) / len(c) * log(1 + avg_words_per_class / total_freq(t))
words_per_class = np.maximum(tf_mat.sum(axis=1, keepdims=True), 1.0)
avg_words = float(words_per_class.mean())
total_freq = np.maximum(tf_mat.sum(axis=0, keepdims=True), 1.0)
ctfidf = (tf_mat / words_per_class) * np.log(1.0 + avg_words / total_freq)

topic_keywords: dict[int, list[str]] = {}
topic_exemplars: dict[int, str] = {}
for k in range(N_TOPICS):
    top_idx = np.argsort(ctfidf[k])[::-1][:8]
    topic_keywords[k] = [str(terms[j]) for j in top_idx]
    member_idx = np.where(cluster_ids == k)[0]
    sims = embeddings[member_idx] @ centroids[k]
    best_doc_idx = int(member_idx[int(np.argmax(sims))])
    topic_exemplars[k] = topic_texts[best_doc_idx][:280]

# 3. Auto-Name Every Topic Using Lightweight Seq2Seq LLM (google/flan-t5-small)
print(f"Generating human-readable topic names with {TOPIC_NAMER_MODEL} ...")
t0 = time.time()
topic_names: dict[int, str] = {}
try:
    t5_tok = AutoTokenizer.from_pretrained(TOPIC_NAMER_MODEL)
    t5_mod = AutoModelForSeq2SeqLM.from_pretrained(TOPIC_NAMER_MODEL).to(DEVICE)
    t5_mod.eval()
    used_names: set[str] = set()

    for k in range(N_TOPICS):
        kw_str = ", ".join(topic_keywords[k][:7])
        prompt = (
            f"Give a concise 2 to 4 word topic title for a group of documents with keywords: {kw_str}. "
            f"Sample text: {topic_exemplars[k][:220]}\nTopic title:"
        )
        inputs = t5_tok(prompt, return_tensors="pt", truncation=True, max_length=256).to(DEVICE)
        with torch.no_grad():
            out_ids = t5_mod.generate(**inputs, max_new_tokens=12, num_beams=4, early_stopping=True)
        raw_name = t5_tok.decode(out_ids[0], skip_special_tokens=True).strip().strip(".,:;\"'")
        # Clean fallback if the small model echoes too few/many chars or duplicates a label
        if len(raw_name) < 3 or len(raw_name) > 48:
            raw_name = " / ".join(w.title() for w in topic_keywords[k][:3])
        clean_name = raw_name.title()
        if clean_name in used_names:
            clean_name = f"{clean_name} ({topic_keywords[k][0].title()})"
        used_names.add(clean_name)
        topic_names[k] = f"T{k:02d}: {clean_name}"

    del t5_mod, t5_tok
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    print(f"[OK] Named all {N_TOPICS} topics in {time.time() - t0:.1f}s")
except Exception as exc:
    print(f"[FALLBACK] Could not load {TOPIC_NAMER_MODEL} ({exc}); using c-TF-IDF keyword titles.")
    for k in range(N_TOPICS):
        topic_names[k] = f"T{k:02d}: " + " / ".join(w.title() for w in topic_keywords[k][:3])

# 4. Build Dataset x Topic Composition Matrix (%) & Summary Table
topic_df = pd.DataFrame({
    "DatasetKey": topic_ds_keys,
    "Dataset": [ds_labels[k] for k in topic_ds_keys],
    "Cluster": cluster_ids,
    "Topic": [topic_names[c] for c in cluster_ids],
})

topic_dist_pct = (
    pd.crosstab(topic_df["Dataset"], topic_df["Topic"], normalize="index") * 100.0
).reindex([ds_labels[k] for k in ds_keys])

topic_catalog_rows = []
for k in range(N_TOPICS):
    t_label = topic_names[k]
    row_info = {
        "Topic": t_label,
        "Docs": int(np.sum(cluster_ids == k)),
        "Top c-TF-IDF Keywords": ", ".join(topic_keywords[k][:7]),
    }
    for dkey in ds_keys:
        row_info[f"{ds_labels[dkey]} (%)"] = round(float(topic_dist_pct.loc[ds_labels[dkey], t_label]), 1)
    topic_catalog_rows.append(row_info)

topic_catalog_df = pd.DataFrame(topic_catalog_rows).sort_values("Docs", ascending=False).reset_index(drop=True)
print("\n=== Discovered BERT Semantic Topics & Dataset Composition (%) ===")
display(topic_catalog_df)

# 5. Visualizations: 2D Semantic Embedding Map + Dataset x Topic Composition Heatmap
pca_2d = PCA(n_components=2, random_state=SEED).fit_transform(embeddings)
topic_palette = sns.color_palette("tab20", n_colors=N_TOPICS)

fig, (ax_map, ax_heat) = plt.subplots(1, 2, figsize=(19.5, 6.8), gridspec_kw={"width_ratios": [1.0, 1.25]})

for k in range(N_TOPICS):
    mask = cluster_ids == k
    ax_map.scatter(pca_2d[mask, 0], pca_2d[mask, 1], s=14, alpha=0.58, color=topic_palette[k], label=topic_names[k])
    cx, cy = pca_2d[mask, 0].mean(), pca_2d[mask, 1].mean()
    ax_map.text(
        cx, cy, f"T{k:02d}", fontsize=8.5, fontweight="bold", color="#0f172a", ha="center", va="center",
        bbox=dict(boxstyle="round,pad=0.2", facecolor="white", edgecolor=topic_palette[k], lw=1.2, alpha=0.92),
    )

ax_map.set_title(f"2D BERT Semantic Topic Map ({BERT_EMBED_MODEL.split('/')[-1]})", fontsize=12, fontweight="bold")
ax_map.set_xlabel("PCA Dimension 1")
ax_map.set_ylabel("PCA Dimension 2")
ax_map.legend(loc="upper left", bbox_to_anchor=(0.0, -0.14), ncol=2, fontsize=8.0, frameon=True)

sns.heatmap(
    topic_dist_pct.T,
    annot=True,
    fmt=".1f",
    cmap="YlGnBu",
    ax=ax_heat,
    cbar_kws={"label": "Share of Dataset (%)"},
)
ax_heat.set_title("Dataset × Topic Composition (%) — Named by flan-t5-small", fontsize=12, fontweight="bold")
ax_heat.set_xlabel("")
ax_heat.set_ylabel("")
ax_heat.tick_params(axis="x", rotation=18)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "03_bert_topic_modeling.png", dpi=150, bbox_inches="tight")
plt.show()

# Free large embedding array
del embeddings, pca_2d, topic_texts
gc.collect()


---
## Section 6 — Domain Proximity (Unigram & Bigram JSD) vs. 13-Gram Contamination

### Separating Healthy Proximity from Benchmark Cheating
* **Unigram & Bigram Jensen-Shannon Divergence (JSD, base-2 bits):** Directly measures distributional distance between each candidate dataset, **OpenWebText** (GPT-2's distribution), and the 4 target benchmarks (`HellaSwag`, `ARC-Easy`, `LAMBADA`, `WikiText-103`). Lower = closer register and phrasing. *(Flat `Eval Word Coverage` was removed because all English corpora clustered tightly around 88–90%.)*
* **13-Gram Exact-Overlap Contamination:** Indexes all normalized alphanumeric 13-grams across **100% of every evaluation split** and flags any verbatim match in our multi-shard candidate slices.

### Visual Guide: Healthy Proximity vs. Verbatim Leakage


In [ ]:
# Uses shared kestrel.utils.notebook helper (local docs/images/ first -> GitHub raw fallback)
show_explainer_visual("04_proximity_vs_contamination.png", "Figure 2: Healthy Domain Proximity (Low JSD, 0% Leakage) vs. Benchmark Cheating (Exact 13-Gram Overlap).")


[Visual: Figure 2: Healthy Domain Proximity (Low JSD, 0% Leakage) vs. Benchmark Cheating (Exact 13-Gram Overlap). — https://raw.githubusercontent.com/MohammedKamalAlsyd/kestrel/master/docs/images/04_proximity_vs_contamination.png]

In [ ]:
def build_ngram_profile(texts: Sequence[str], top_vocab_k: int = 10_000, top_bigram_k: int = 50_000) -> dict[str, Any]:
    unigrams: Counter[str] = Counter()
    bigrams: Counter[str] = Counter()
    for txt in texts:
        words = normalize_words(txt)
        unigrams.update(words)
        if len(words) >= 2:
            bigrams.update(f"{words[i]} {words[i + 1]}" for i in range(len(words) - 1))
    return {
        "top_vocab": {w for w, _ in unigrams.most_common(top_vocab_k)},
        "unigram_counts": dict(unigrams),
        "bigram_counts": dict(bigrams.most_common(top_bigram_k)),
    }


def compute_jsd(counts_a: dict[str, int], counts_b: dict[str, int]) -> float:
    if not counts_a or not counts_b:
        return 1.0
    keys = list(set(counts_a.keys()) | set(counts_b.keys()))
    p = np.array([counts_a.get(k, 0) for k in keys], dtype=np.float64)
    q = np.array([counts_b.get(k, 0) for k in keys], dtype=np.float64)
    if p.sum() == 0.0 or q.sum() == 0.0:
        return 1.0
    p /= p.sum()
    q /= q.sum()
    d = float(jensenshannon(p, q, base=2.0))
    return 1.0 if math.isnan(d) else float(min(1.0, max(0.0, d * d)))


def audit_domain_alignment(cand_prof: dict[str, Any], ref_prof: dict[str, Any]) -> dict[str, float]:
    union_sz = len(cand_prof["top_vocab"] | ref_prof["top_vocab"])
    jaccard = float(len(cand_prof["top_vocab"] & ref_prof["top_vocab"]) / union_sz) if union_sz else 0.0
    return {
        "top10k_jaccard": jaccard,
        "unigram_jsd": compute_jsd(cand_prof["unigram_counts"], ref_prof["unigram_counts"]),
        "bigram_jsd": compute_jsd(cand_prof["bigram_counts"], ref_prof["bigram_counts"]),
    }


def build_eval_13gram_index(eval_texts: Sequence[str], n: int = 13) -> tuple[set[str], dict[str, list[int]], int]:
    all_ngs: set[str] = set()
    ng_to_ex: dict[str, list[int]] = defaultdict(list)
    eligible = 0
    for idx, txt in enumerate(eval_texts):
        ngs = extract_ngrams_set(txt, n=n)
        if not ngs:
            continue
        eligible += 1
        all_ngs.update(ngs)
        for ng in ngs:
            ng_to_ex[ng].append(idx)
    return all_ngs, ng_to_ex, eligible


def audit_contamination_13gram(cand_texts: Sequence[str], eval_index: tuple[set[str], dict[str, list[int]], int], n: int = 13) -> dict[str, Any]:
    eval_ngs, ng_to_ex, eligible = eval_index
    n_docs = len(cand_texts)
    if n_docs == 0 or not eval_ngs:
        return {
            "eligible_eval_examples": eligible, "contaminated_candidate_docs": 0,
            "contaminated_eval_examples": 0, "eval_contamination_rate": 0.0,
            "matched_unique_ngrams": 0, "sample_matches": [],
        }
    hit_docs = 0
    matched: set[str] = set()
    hit_eval: set[int] = set()
    for txt in cand_texts:
        words = normalize_alnum_words(txt)
        if len(words) < n:
            continue
        doc_hit = False
        for i in range(len(words) - n + 1):
            ng = " ".join(words[i : i + n])
            if ng in eval_ngs:
                doc_hit = True
                matched.add(ng)
                hit_eval.update(ng_to_ex[ng])
        if doc_hit:
            hit_docs += 1
    return {
        "eligible_eval_examples": eligible,
        "contaminated_candidate_docs": hit_docs,
        "contaminated_eval_examples": len(hit_eval),
        "eval_contamination_rate": float(len(hit_eval) / eligible) if eligible else 0.0,
        "matched_unique_ngrams": len(matched),
        "sample_matches": sorted(matched)[:5],
    }


# Run Alignment & 13-Gram Contamination Audits
t0 = time.time()
cand_profiles = {k: build_ngram_profile([r["text"] for r in candidate_slices[k]]) for k in ds_keys}
ref_profiles = {}
if "openwebtext" in cand_profiles:
    ref_profiles["openwebtext"] = cand_profiles["openwebtext"]
eval_13gram_indices = {}
for bkey, btexts in eval_texts_by_benchmark.items():
    ref_profiles[bkey] = build_ngram_profile(btexts)
    eval_13gram_indices[bkey] = build_eval_13gram_index(btexts, n=13)

alignment_matrix = {
    ck: {rk: audit_domain_alignment(cand_profiles[ck], rprof) for rk, rprof in ref_profiles.items()}
    for ck in ds_keys
}
contamination_matrix = {
    ck: {bk: audit_contamination_13gram([r["text"] for r in candidate_slices[ck]], eval_13gram_indices[bk]) for bk in eval_texts_by_benchmark}
    for ck in ds_keys
}
print(f"[DONE] Alignment & 13-Gram Contamination audits finished in {time.time() - t0:.1f}s")

# Plot 3-Panel Heatmaps: Unigram JSD, Bigram JSD, and 13-Gram Contamination (%)
ref_keys = (["openwebtext"] if "openwebtext" in ref_profiles else []) + list(eval_texts_by_benchmark.keys())
ref_labels = {
    "openwebtext": "OpenWebText\n(GPT-2 Ctrl)",
    **{k: v.display_name.replace(" ", "\n") for k, v in EVAL_BENCHMARKS.items()},
}

uni_jsd_df = pd.DataFrame({ref_labels[rk]: [alignment_matrix[dk][rk]["unigram_jsd"] for dk in ds_keys] for rk in ref_keys}, index=[ds_labels[dk] for dk in ds_keys])
bi_jsd_df = pd.DataFrame({ref_labels[rk]: [alignment_matrix[dk][rk]["bigram_jsd"] for dk in ds_keys] for rk in ref_keys}, index=[ds_labels[dk] for dk in ds_keys])
contam_df = pd.DataFrame({ref_labels[bk]: [100.0 * contamination_matrix[dk][bk]["eval_contamination_rate"] for dk in ds_keys] for bk in eval_texts_by_benchmark}, index=[ds_labels[dk] for dk in ds_keys])

fig, axes = plt.subplots(1, 3, figsize=(20.5, 0.85 * len(ds_keys) + 3.2))

sns.heatmap(uni_jsd_df, annot=True, fmt=".3f", cmap="YlOrRd", ax=axes[0], cbar_kws={"label": "Unigram JSD (bits, lower = closer)"})
axes[0].set_title("Unigram Jensen-Shannon Divergence\n(Lower = Closer Vocabulary Distribution)", fontweight="bold")

sns.heatmap(bi_jsd_df, annot=True, fmt=".3f", cmap="YlOrRd", ax=axes[1], cbar_kws={"label": "Bigram JSD (bits, lower = closer)"})
axes[1].set_title("Bigram Jensen-Shannon Divergence\n(Lower = Closer Phrasing & Syntax)", fontweight="bold")
axes[1].set_yticklabels([])

sns.heatmap(contam_df, annot=True, fmt=".3f", cmap="Reds", vmin=0.0, ax=axes[2], cbar_kws={"label": "Eval Examples Hit (%)"})
axes[2].set_title("13-Gram Exact-Overlap Contamination (%)\n(Target: ~0.0% Verbatim Leakage)", fontweight="bold")
axes[2].set_yticklabels([])

plt.tight_layout()
plt.savefig(FIGURES_DIR / "04_jsd_and_contamination_heatmaps.png", dpi=150, bbox_inches="tight")
plt.show()


---
## Section 7 — Final Ranked Scorecard & Kestrel's 2-Stage WSD Mixture

### Length-Unbiased Composite Score (0–100)
With length-confounded **LZ4** and flat **Eval Word Coverage** removed, every point of the scorecard reflects real, non-overlapping signal:
* **40% Cleanliness (`1 - Noise Rate`):** `Noise Rate = 0.70 * Gopher Repetition Flag + 0.30 * Cross-Doc 7-Gram Boilerplate`.
* **45% GPT-2 Benchmark Alignment:** `100 * (0.60 * (1 - Mean Eval Unigram JSD) + 0.40 * (1 - Mean Eval Bigram JSD))` across `HellaSwag`, `ARC-Easy`, `LAMBADA`, and `WikiText-103`.
* **15% Decontamination Safety:** Penalizes datasets as max 13-gram eval overlap approaches 5%.

### Why We Blend Instead of Training 100% on Rank #1
As our **BERT Topic Composition** and **JSD Heatmaps** show, `DCLM-Baseline` anchors everyday language & narrative fluency (`LAMBADA`, `HellaSwag`), while `FineWeb-Edu` and `FinePDFs-Edu` supply the dense STEM & long-form expository reasoning (`ARC-Easy`, `WikiText-103`) a 350M–700M model needs to beat 1.5B GPT-2 XL. Kestrel therefore uses a **Two-Stage WSD Curriculum** (90% diverse base + 10% high-density annealing with a 15% `DCLM-Baseline` anti-forgetting anchor):


In [ ]:
# Uses shared kestrel.utils.notebook helper (local docs/images/ first -> GitHub raw fallback)
show_explainer_visual("anim_02_wsd_curriculum_mixture.gif", "Animation 2: Kestrel's Two-Stage WSD Curriculum (90% Base Mix → 10% High-Density Annealing) vs. GPT-2 XL.")


[Visual: Animation 2: Kestrel's Two-Stage WSD Curriculum (90% Base Mix → 10% High-Density Annealing) vs. GPT-2 XL. — https://raw.githubusercontent.com/MohammedKamalAlsyd/kestrel/master/docs/images/anim_02_wsd_curriculum_mixture.gif]

In [ ]:
scorecard_rows = []
for key in ds_keys:
    recs = candidate_slices[key]
    qa = quality_audits[key]
    noise_rate = 0.70 * qa["gopher_flag_rate"] + 0.30 * qa["cross_doc_boilerplate_doc_rate"]

    eval_aligns = [v for rk, v in alignment_matrix[key].items() if rk != "openwebtext"]
    mean_uni_jsd = float(np.mean([a["unigram_jsd"] for a in eval_aligns]))
    mean_bi_jsd = float(np.mean([a["bigram_jsd"] for a in eval_aligns]))
    mean_jacc = float(np.mean([a["top10k_jaccard"] for a in eval_aligns]))
    align_score = 100.0 * (0.60 * (1.0 - mean_uni_jsd) + 0.40 * (1.0 - mean_bi_jsd))

    owt_jsd = alignment_matrix[key].get("openwebtext", {}).get("unigram_jsd", float("nan"))
    mean_contam = 100.0 * float(np.mean([c["eval_contamination_rate"] for c in contamination_matrix[key].values()]))
    max_contam = 100.0 * float(np.max([c["eval_contamination_rate"] for c in contamination_matrix[key].values()]))

    composite = (
        40.0 * (1.0 - min(1.0, noise_rate))
        + 45.0 * (align_score / 100.0)
        + 15.0 * max(0.0, 1.0 - max_contam / 5.0)
    )
    total_tok = sum(r["n_tokens"] for r in recs)
    scorecard_rows.append({
        "Dataset": ds_labels[key],
        "Key": key,
        "Docs": len(recs),
        "Tokens": total_tok,
        "Mean Tok/Doc": round(total_tok / max(len(recs), 1), 1),
        "Gopher Rep Flag (%)": round(100.0 * qa["gopher_flag_rate"], 2),
        "Cross-Doc Boilerplate (%)": round(100.0 * qa["cross_doc_boilerplate_doc_rate"], 2),
        "Noise Rate (%)": round(100.0 * noise_rate, 2),
        "OWT Unigram JSD": round(owt_jsd, 4) if not math.isnan(owt_jsd) else None,
        "Eval Mean JSD (Uni)": round(mean_uni_jsd, 4),
        "Eval Mean JSD (Bi)": round(mean_bi_jsd, 4),
        "GPT-2 Benchmark Alignment": round(align_score, 2),
        "13-gram Contam Mean (%)": round(mean_contam, 4),
        "13-gram Contam Max (%)": round(max_contam, 4),
        "Composite Score": round(composite, 2),
    })

scorecard = pd.DataFrame(scorecard_rows).sort_values("Composite Score", ascending=False).reset_index(drop=True)
scorecard.insert(0, "Rank", np.arange(1, len(scorecard) + 1))

display_cols = [c for c in scorecard.columns if c != "Key"]
print("=== Kestrel Pretraining Dataset Audit Scorecard (Length-Unbiased) ===")
styled = (
    scorecard[display_cols]
    .style.background_gradient(subset=["Composite Score", "GPT-2 Benchmark Alignment"], cmap="Greens")
    .background_gradient(subset=["Noise Rate (%)", "13-gram Contam Max (%)"], cmap="Reds")
    .format(precision=3, na_rep="—")
)
display(styled)

# Transparently inspect any exact 13-gram overlaps
print("\n=== Sample 13-Gram Exact Matches (Human Contamination Inspection) ===")
any_hits = False
for dk in ds_keys:
    for bk, cres in contamination_matrix[dk].items():
        if cres["matched_unique_ngrams"] > 0:
            any_hits = True
            print(
                f"  - {ds_labels[dk]} x {bk}: {cres['contaminated_eval_examples']}/{cres['eligible_eval_examples']} "
                f"eval examples ({100 * cres['eval_contamination_rate']:.3f}%), "
                f"{cres['contaminated_candidate_docs']} candidate docs"
            )
            for snippet in cres["sample_matches"][:3]:
                print(f'      match: "{snippet}"')
if not any_hits:
    print("  [CLEAN] Zero exact 13-gram overlaps detected across all audited slices.")

# Export Artifacts for Kestrel Training Logs
scorecard.to_csv(SAMPLES_DIR / "scorecard.csv", index=False)
topic_catalog_df.to_csv(SAMPLES_DIR / "topic_distribution.csv", index=False)
(SAMPLES_DIR / "audit_summary.json").write_text(
    json.dumps({
        "config": {
            "target_tokens": TARGET_TOKENS, "max_docs": MAX_DOCS,
            "random_sample": RANDOM_SAMPLE, "num_shards": NUM_SHARDS_TO_SAMPLE,
            "bert_model": BERT_EMBED_MODEL, "topic_namer": TOPIC_NAMER_MODEL, "n_topics": N_TOPICS,
        },
        "scorecard": scorecard.to_dict(orient="records"),
        "alignment_matrix": alignment_matrix,
        "contamination_summary": {
            dk: {bk: {k: v for k, v in cres.items() if k != "sample_matches"} for bk, cres in bm.items()}
            for dk, bm in contamination_matrix.items()
        },
    }, indent=2),
    encoding="utf-8",
)

print(f"\nSaved scorecard CSV       : {SAMPLES_DIR / 'scorecard.csv'}")
print(f"Saved topic distribution  : {SAMPLES_DIR / 'topic_distribution.csv'}")
print(f"Saved audit summary JSON  : {SAMPLES_DIR / 'audit_summary.json'}")
print(f"Saved figures under       : {FIGURES_DIR}")
